# 01. Signal processing

Processes Q00 recordings with C01 and C02 stored separately. `baseline` and `comparison` are analytical roles and do not confirm irradiation. Frequencies come from the provided catalogue table. Metadata is edited only in `src/dataset_config.py`.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image

cwd = Path.cwd().resolve()
BASE_DIR = next((p for p in [cwd, *cwd.parents] if (p / "src" / "dataset_config.py").exists()), None)
if BASE_DIR is None:
    raise RuntimeError("Open this notebook from the project directory or notebooks subdirectory")
if str(BASE_DIR / "src") not in sys.path:
    sys.path.insert(0, str(BASE_DIR / "src"))
import pandas as pd
from dataset_config import DATA_DIR, DATASET_PATH, FIGURES_DIR, FILE_METADATA
print(BASE_DIR)

In [ ]:
from signal_processing import (find_input_file, read_q00_file, iter_samples,
                               build_combined_dataset)
metadata = pd.DataFrame.from_dict(FILE_METADATA, orient="index")
metadata.index.name = "file"
display(metadata)

In [ ]:
availability = metadata.reset_index()
availability["available"] = availability["file"].map(lambda stem: find_input_file(DATA_DIR, stem) is not None)
display(availability[["file", "experiment_id", "channel", "analysis_role", "available"]])

## Single example

Reads only the first record, without loading the whole file into memory. `sample_start` is the index of a point within the record, not absolute time.

In [ ]:
available = availability.loc[availability["available"], "file"]
if available.empty:
    raise FileNotFoundError("No registered Q00 files found")
example_stem = "BI202C01" if "BI202C01" in set(available) else available.iloc[0]
reader = read_q00_file(find_input_file(DATA_DIR, example_stem))
try:
    record = next(reader)
finally:
    reader.close()
print(example_stem, {key: value for key, value in record.items() if key != "codes"})
print("Points:", len(record["codes"]))
first_window = next(iter_samples(record["codes"]), None)
print("First window start / length:", None if first_window is None else (first_window[0], len(first_window[1])))

## Build or load

Set `REBUILD = True` after changing metadata or parameters. Processing large files can take a while.

In [ ]:
REBUILD = False
if REBUILD or not DATASET_PATH.exists():
    df = build_combined_dataset()
else:
    df = pd.read_csv(DATASET_PATH)
required = {"material", "channel", "experiment_id", "analysis_role", "control_mode",
            "measurement_type", "electrode_distance_mm", "record_index", "sample_start"}
if required - set(df.columns):
    raise ValueError("Old CSV: set REBUILD=True and rerun this cell")
display(df.head())

In [ ]:
display(df.groupby(["material", "experiment_id", "channel"]).size().rename("n_windows").to_frame())
display(df.groupby(["material", "experiment_id", "channel"])[["gamma", "log_energy"]].agg(["count", "mean", "std"]))

## Plots

These use the same functions as `train_models.py`. Running this cell overwrites the four feature figures in `results/figures`. Panels have independent y-axis scales. Error bars show the standard deviation across windows, not a confidence interval.

In [ ]:
from train_models import load_dataset, save_feature_plots
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
clean_df = load_dataset()
save_feature_plots(clean_df)
for filename in ("file_level_gamma_comparison.png", "file_level_energy_comparison.png",
                 "gamma_boxplots_by_file.png", "energy_boxplots_by_file.png"):
    if (FIGURES_DIR / filename).exists():
        display(Image(filename=str(FIGURES_DIR / filename)))

## Limitations

`gamma` is a signed slope. `energy` is the mean of the unnormalized squared FFT magnitude, not a PSD. Changing the frequency without changing the frequency indices used does not correct the slope. Reference channels and exploratory series are registered but not used for training automatically. For COB01, electrode distance is labelled only for C03, according to the provided description.